## We're going to wrap getting the data, training the FNO, and running inference into functions so that we can use them for the ablation study

In [ ]:
# Imports
import numpy as np
import sys

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern

import torch
from torch.utils.data import DataLoader

import neuralop
from neuralop.models import FNO
from neuralop import Trainer, LpLoss, H1Loss
from neuralop.training import AdamW
from neuralop.data.datasets import load_darcy_flow_small, DarcyDataset
from neuralop.utils import count_model_params
from torch.nn import functional as F

import matplotlib.pyplot as plt

In [ ]:
# HELPERS
def T_push(x, tau=-1, boundary=0):
    """A smooth approximation to the step function, which can be used to get a binary field from the GP sample while maintaining differentiability.
    See Akiyldiz for details.
    Combined with the option to get a hard threshold by setting tau=-1, based on the usual 3-12 normalisation.
    """
    if isinstance(x, np.ndarray):
        if tau == -1:
            return np.where(x < boundary, 3.0, 12.0)
        return 0.5 * np.tanh(tau * x) + 0.5
    else:  # torch tensor
        if tau == -1:
            return torch.where(x < boundary,
                               torch.tensor(3.0),
                               torch.tensor(12.0))
        return 0.5 * torch.tanh(tau * x) + 0.5

In [ ]:
def get_data(N_train, N_test, resolution_TRAIN, resolution_TEST, data_batch_size=32, data_test_batch_sizes=[32, 32])
    # Import Data
    # Loading the Darcy-Flow dataset

    dataset = DarcyDataset(
        root_dir="./darcy_data",
        n_train=N_train,
        n_tests=N_test,
        train_resolution=resolution_TRAIN,
        batch_size=data_batch_size,
        test_resolutions=resolution_TEST,
        test_batch_sizes=data_test_batch_sizes,
    )

    # renormalise data to 3-12 range, as per convention. For some reason the default normalisation is 0-1 which is not the norm
    # only run the code ONCE, otherwise it will normalise twice and all entries become 12.
    for test_set in dataset.test_dbs.values():
        test_set.x = T_push(test_set.x, tau=-1, boundary=0.5)

    dataset._train_db.x = T_push(dataset._train_db.x, tau=-1, boundary=0.5)


    train_loader = DataLoader(
            dataset.train_db,
            batch_size=data_batch_size,
            num_workers=1,
            pin_memory=True,
            persistent_workers=False,
        )

    test_loaders = {}
    for res, test_bsize in zip(resolution_TEST, data_test_batch_sizes):
        test_loaders[res] = DataLoader(
            dataset.test_dbs[res],
            batch_size=test_bsize,
            shuffle=False,
            num_workers=1,
            pin_memory=True,
            persistent_workers=False,
        )

    data_processor = dataset.data_processor
    return data_processor



In [ ]:
def train_FNO(FNO_n_modes, FNO_hidden_channels, FNO_n_layers, FNO_nonlinearity, FNO_factorization="tucker", FNO_in_channels=1, FNO_out_channels=1):
    # Instantiate Model
    model = FNO(
        n_modes=FNO_n_modes,
        in_channels=FNO_in_channels,
        out_channels=FNO_out_channels,
        hidden_channels=FNO_hidden_channels,
        n_layers=FNO_n_layers,
        # projection_channel_ratio=FNO_projection_channel_ratio,
        factorization=FNO_factorization,
        non_linearity=FNO_nonlinearity,
        # rank=FNO_rank,
    )

    # NOTE: FNO takes in inputs of shape (batch_size, in_channels, height, width) and outputs the same shape.
    # If you want to pass a single observation, you need to add batch and channel dimensions, e.g., x[0, 0].unsqueeze(0).unsqueeze(0) to get shape (1, 1, 16, 16).

    model = model.to(device)

    n_params = count_model_params(model)
    print(f"\nOur model has {n_params} parameters.")
    sys.stdout.flush()


    # Scheduler and Loss

    optimizer = AdamW(model.parameters(), lr=8e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

    l2loss = LpLoss(d=2, p=2)
    h1loss = H1Loss(d=2)

    train_loss = h1loss
    eval_losses = {"h1": h1loss, "l2": l2loss}

    # Train model
    # Creating the trainer
    trainer = Trainer(
        model=model,
        n_epochs=TRAIN_epochs,
        device=device,
        data_processor=data_processor,
        wandb_log=False,
        eval_interval=5,
        use_distributed=False,
        verbose=True,
    )

    # DELETE checkpoints FOLDER BEFORE TRAINING FROM SCRATCH
    # We train and save checkpoints
    # Train from scratch
    trainer.train(
        train_loader=train_loader,
        test_loaders={},
        optimizer=optimizer,
        scheduler=scheduler,
        regularizer=False,
        training_loss=train_loss,
        save_every=5,
        save_dir="./checkpoints",
    )

    return model


In [ ]:
def run_MCMC()